In [66]:
from brian2 import *
import os, sys
root = os.path.dirname(os.getcwd())  # go up from 'Interactive Models'
models_dir = os.path.join(root, 'Neuron and Synapse Models')
tools_dir = os.path.join(root, 'Tools')
for p in (models_dir, tools_dir):
    if p not in sys.path:
        sys.path.append(p)

from neuronModels import *
from ringAttractorClass import *
from ringAttractorClass import RingAttractor
from plottingTools import *
from faithfulRingAttractorClass import FaithfulRingAttractor
from utils import computePVA


import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, IntSlider, Dropdown, HTML, FloatText, Label, ToggleButton

# Simulation parameters
defaultclock.dt = 0.1*ms

In [67]:
# Code block to allow easy passing of intital values
values_flag = True
if values_flag:
    init_values = {
        'num_neurons': 12,
        'tau_val': 10,
        'tau_syn_val': 13,  # synaptic decay (ms)
        'sigma_noise_val': 0.1,
        'stimulus_center': 3.14,
        'stimulus_width': 0.1,
        'I0_val': 10,
        'g_cosine_val': 18.42230909, #10.333033257636599,  # mV gain amplitude (cosine)
        'w_inh_val':  21.72109356,#10.333033268750986,      # mV subtractive inhibition magnitude (positive number)
        'velocity_input': 0.0,
        'input_duration': 1.0,
        'duration_val': 1.0,
        'velocity_duration_val': 3.0,
        'Iff_val': 80.0  # mA
    }

In [68]:
# Create sliders for parameters
num_neurons_slider = IntSlider(
    min=4,
    max=200, 
    step=1, 
    value=init_values['num_neurons'] if values_flag else 120, 
    description='Number of Neurons:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

# Membrane tau (ms)
tau_slider = FloatSlider(
    min=1, 
    max=20, 
    step=1, 
    value=init_values['tau_val'] if values_flag else 10, 
    description='Tau (ms):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

# Synaptic decay tau_s (ms)
tau_syn_slider = FloatSlider(
    min=0.0,
    max=50.0,
    step=0.5,
    value=init_values['tau_syn_val'] if values_flag else 13.0,
    description='Tau synaptic (ms):',
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_noise_slider = FloatSlider(
    min=0.0, 
    max=5, 
    step=0.1, 
    value=init_values['sigma_noise_val'] if values_flag else 1, 
    description='Noise Sigma (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_center_slider = FloatSlider(
    min=0, 
    max=2*pi, 
    step=0.01, 
    value=init_values['stimulus_center'] if values_flag else 0, 
    description='Stimulus Center (rad):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_width_slider = FloatSlider(
    min=0.1, 
    max=2.0, 
    step=0.01, 
    value=init_values['stimulus_width'] if values_flag else 0.5, 
    description='Stimulus Width:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

I0_slider = FloatSlider(
    min=0, 
    max=150, 
    step=5, 
    value=init_values['I0_val'] if values_flag else 30, 
    description='Input Amplitude (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

# Constant feed-forward current (mA)
Iff_slider = FloatSlider(
    min=0.0,
    max=120.0,
    step=1.0,
    value=init_values['Iff_val'] if values_flag else 80.0,
    description='I_ff (mA):',
    continuous_update=False,
    style={'description_width': '150px'},
)

# Cosine gain (mV)
g_cosine_slider = FloatSlider(
    min=0.0,
    max=500.0,
    step=0.001,
    value=init_values['g_cosine_val'] if values_flag else 10.0,
    description='Cosine gain (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
    readout_format='.2f'
)

# Subtractive inhibitory weight (mV), positive magnitude and applied subtractively
w_inh_slider = FloatSlider(
    min=0.0, 
    max=500.0, 
    step=0.001, 
    value=init_values['w_inh_val'] if values_flag else 5.0, 
    description='Inhibitory weight (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
    readout_format='.2f'
)

velocity_input_slider = FloatSlider(
    min=-8.0, 
    max=8.0, 
    step=0.1, 
    value=init_values['velocity_input'] if values_flag else 0.0, 
    description='Velocity Input (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)



# durations
duration_box = FloatText(
    value=init_values['duration_val'] if values_flag else 2,
    description='In-Between Duration (s):',
    style={'description_width': '150px'}
)

input_duration_box = FloatText(
    value=init_values['input_duration'] if values_flag else 1,
    description='Input Duration (s):',
    style={'description_width': '150px'}
)

velocity_duration_box = FloatText(
    value=init_values['velocity_duration_val'] if values_flag else 0.5,
    description='Velocity Duration (s):',
    style={'description_width': '150px'}
)

angles_dropdown = Dropdown(
    options=[ 'degrees', 'radians', 'radians (symbolic)'],
    value='degrees',
    description='Angular Representation:',
    style={'description_width': '150px'},
)

autapse_button = ToggleButton(
    value = True,
    description='Autapse',
    tooltip='Allows autapse connections',
    button_style=''
)

# Create dictionary of widgets
widgets = {
    'num_neurons': num_neurons_slider,
    'tau_val': tau_slider,
    'tau_syn_val': tau_syn_slider,
    'sigma_noise_val': sigma_noise_slider,
    'stimulus_center': stimulus_center_slider,
    'stimulus_width': stimulus_width_slider,
    'I0_val': I0_slider,
    'g_cosine_val': g_cosine_slider,
    'duration_val': duration_box,
    'input_duration_val': input_duration_box,
    'velocity_duration_val': velocity_duration_box,
    'ticks_angles': angles_dropdown,
    'autapse': autapse_button,
    'velocity_input': velocity_input_slider,
    'w_inh_val': w_inh_slider,
    'Iff_val': Iff_slider,
}

In [ ]:
import numpy as np
import os

# Updated interactive_simulator to accept velocity_duration_val, tau_synaptic, and I_ff

def interactive_simulator(num_neurons, tau_val, sigma_noise_val,
                          stimulus_center, stimulus_width, I0_val, 
                          g_cosine_val, velocity_input, w_inh_val,
                          duration_val, input_duration_val, velocity_duration_val,
                          autapse, ticks_angles,
                          tau_synaptic_val, Iff_val):
    
    # Clear any previous figures
    plt.close('all')
    
    # Convert slider values to Brian units
    tau = tau_val * ms
    tau_s = tau_synaptic_val * ms
    sigma_noise = sigma_noise_val * mV
    V_rest = -70 * mV
    I0 = I0_val * mV
    sim_duration = duration_val*second
    g_cosine = g_cosine_val * mV
    w_inh_v = w_inh_val * mV  # positive magnitude; applied subtractively inside FaithfulRingAttractor
    # Map mA (slider) to voltage-equivalent via R=1 Ω to match neuron equation units
    I0_CONST = (Iff_val * mA) * ohm
    
    velocity_duration = velocity_duration_val  # seconds
    
    # Define neuron positions
    positions = linspace(0, 2*pi, num_neurons, endpoint=False)
    
    # Calculate external input
    d = np.angle(np.exp(1j * (positions - stimulus_center)))
    I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width**2)) + I0_CONST
    
    # Set up neuron model
    neuron_eq = Equations(LIF_synapticDecay_xi_vel_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise, tau_s=tau_s)
    
    # Set up ring attractor
    Vth = -48 * mV
    V_reset = -80 * mV
    refractory_period = 5 * ms
    
    # Create the ring attractor network
    ringAttractor = FaithfulRingAttractor(neuron_eq, 
                        num_neurons, 
                        Vth, V_reset, refractory_period,
                        autapse=autapse, profile='cosine',
                        w_sub=w_inh_v, normalized=False,
                        g_cosine=g_cosine, g_sine=1*mV)
    # Set external input
    ringAttractor.ring_pool.I_ext = I_ext_array
    ringAttractor.ring_pool.I_vel = 0.0*volt

        
    # Setup monitors
    spikemon = SpikeMonitor(ringAttractor.ring_pool)
    statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
    inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)
    isynmon = StateMonitor(ringAttractor.ring_pool, 'I_syn', record=True)
    
    #+---------------------------------------------------------------------------+
    #|                             Network Operations                            |
    #+---------------------------------------------------------------------------+
    # Clipping - Reverse Potential Behaviour: Define a network operation to enforce the lower bound on the membrane potential
    @network_operation(dt=defaultclock.dt)
    def enforce_lower_bound():
        # Using the built-in clip function (from numpy)
        ringAttractor.ring_pool.V[:] = clip(ringAttractor.ring_pool.V[:], V_reset, inf*volt)
    
    # Set of Brian objects to be added to the network
    localObjects = [enforce_lower_bound,
                    spikemon, statemon, inputmon, isynmon]
    
    net = Network(ringAttractor.BrianObjects+localObjects)
    
    input_on = input_duration_val * second
    input_off = sim_duration
    velocity_on = velocity_duration * second
    end_duration = sim_duration
    
    total_duration = input_on + input_off + velocity_on + end_duration

    # Run simulation
    net.run(input_on)
    
    # Turn off input for the second half
    ringAttractor.ring_pool.I_ext = I0_CONST
    net.run(input_off)
    
    # Turn on velocity input
    ringAttractor.ring_synapses_asym.vel_in = velocity_input
    ringAttractor.ring_synapses_asym.vel_on = True
      
    net.run(velocity_on)
    
    # Turn off velocity input
    ringAttractor.ring_synapses_asym.vel_in = 0.0
    ringAttractor.ring_synapses_asym.vel_on = False
    
    # Turn off velocity input and run for the rest of the duration
    net.run(end_duration)
    
    #+---------------------------------------------------------------------------+
    #|                           Plotting the Results                            |
    #+---------------------------------------------------------------------------+
    # Use a 3x2 grid so the time-resolved PVA has its own row slot.
    fig = plt.figure(figsize=(14, 14))
    
    # 1. Input Current Plot (row1 col1)
    ax1 = fig.add_subplot(3, 2, 1)
    ax1.plot(positions/(2*pi), I_ext_array/mV)
    ax1.set_title('Input Current')
    ax1.set_xlabel('Position (fraction of 2π)')
    ax1.set_ylabel('Current (mV)')
    
    # 2. Raster Plot (row1 col2)
    ax2 = fig.add_subplot(3, 2, 2)
    ax2.axvspan((input_on+input_off)/second, (input_on+velocity_on+input_off)/second, color='green', alpha=0.2, label='Velocity Input ON')

    raster_plot(spikemon, ax=ax2, stim_periods=(0*second, input_on),
                stim_display_method='highlight', duration=total_duration, num_neurons=num_neurons, y_axisFull=True)
    ax2.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), fontsize=8)
    
    # 3. Firing Rate Profile Plot (row2 col1)
    ax3 = fig.add_subplot(3, 2, 3)
    start_mask=(input_on+input_off+velocity_on+input_off)/2
    mask=spikemon.t>start_mask
    reduced_duration=total_duration-start_mask
    class ReducedMonitor:
        def __init__(self, t=None, i=None):
            self.t = t
            self.i = i
    reduced_monitor = ReducedMonitor(t=spikemon.t[mask], i=spikemon.i[mask])
    firing_rate, _ = firing_rate_profile(reduced_monitor, positions/(2*pi), reduced_duration, ax=ax3)
    
    # 4. Polar Plot of PVA (row2 col2)
    ax4 = fig.add_subplot(3, 2, 4, projection='polar')
    polar_plot_PVA(firing_rate, positions, scale=1.2, ax=ax4)
    pva_angle, pva_magnitude = computePVA(firing_rate, positions)
    ax4.set_rlim(0, second*1/refractory_period)
    ax4.plot([pva_angle,pva_angle], [0,pva_magnitude], 'cyan', label=f'PVA: ({pva_angle*180/pi:.2f}, {pva_magnitude:.2f})')
    ax4.legend(loc='upper right', bbox_to_anchor=(1.15, 1.1), fontsize=8)
    
    # 5. Time-Resolved PVA Plot (row3 spans both columns)
    ax5 = fig.add_subplot(3, 1, 3)  # full-width axis on third row
    _ , _ = time_resolved_PVA(spikemon, positions, total_duration, num_neurons, window_size=100*ms,
                              ax=ax5, color_windows=False, stim_periods=(0*second, input_on))
    ax5.set_title('Time-Resolved PVA')
    ax5.set_ylabel('Angle (rad)')
    # Ensure full angular range visible
    ax5.set_ylim(0, 2 * pi)

    # Dotted horizontal lines at neuron angles, colored by angle (HSV)
    x0, x1 = 0.0, total_duration/second
    cmap = plt.cm.hsv
    for ang in positions:
        col = cmap(((ang % (2*pi)) / (2*pi)))
        ax5.hlines(y=ang, xmin=x0, xmax=x1, colors=[col], linestyles=':', linewidth=1.1, alpha=0.8)

    # Green vertical band when velocity input is active
    t_vel_on = (input_on + input_off)/second
    t_vel_off = (input_on + input_off + velocity_on)/second
    ax5.axvspan(t_vel_on, t_vel_off, color='green', alpha=0.2, label='Velocity Input ON')

    # Consolidate legend (avoid duplicates)
    handles, labels = ax5.get_legend_handles_labels()
    seen = {}
    uniq_h, uniq_l = [], []
    for h, l in zip(handles, labels):
        if l not in seen:
            seen[l] = True
            uniq_h.append(h)
            uniq_l.append(l)
    ax5.legend(uniq_h, uniq_l, loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()

    # Plot I_syn heatmap over time (separate figure to keep layouts clean)
    plt.figure(figsize=(12, 4))
    # isynmon.I_syn has shape (num_neurons, time_points)
    im = plt.imshow((isynmon.I_syn/mV), aspect='auto', origin='lower',
                    extent=[isynmon.t[0]/second, isynmon.t[-1]/second, 0, num_neurons],
                    cmap='viridis')
    plt.colorbar(im, label='I_syn (mV)')
    plt.xlabel('Time (s)')
    plt.ylabel('Neuron index')
    plt.title('Synaptic current I_syn over time')
    plt.tight_layout()
    plt.show()
    
    
    
    

In [70]:
# # Define common style and layout settings for sliders
# common_style = {'description_width': '150px'}
# common_layout = Layout(width='300px', margin='10px auto')

# Create the parameter boxes with descriptive titles
from ipywidgets import Button, Output  # add missing widgets

box_neurons = VBox([
    HTML(value="<b>Neurons Parameters:</b>"),
    num_neurons_slider,
    tau_slider,
    tau_syn_slider,
    sigma_noise_slider,
    Iff_slider,
], layout=Layout(width='25%', align_items='center'))

box_input = VBox([
    HTML(value="<b>Input Parameters:</b>"),
    stimulus_center_slider,
    stimulus_width_slider,
    I0_slider,
    velocity_input_slider
], layout=Layout(width='25%', align_items='center'))

checkbox_subBox = HBox([
    autapse_button,
], layout=Layout(align_items='center'))

box_connectivity = VBox([
    HTML(value="<b>Connectivity Profile Parameters:</b>"),
    checkbox_subBox,
    g_cosine_slider,
    w_inh_slider,
], layout=Layout(width='25%', align_items='center'))

# (Currently only g_cosine and w_inh are relevant; logic placeholder if more profiles reintroduced.)
def update_connectivity_box(*args):
    # Always show cosine gain slider and inhibitory weight slider
    box_connectivity.children = [
        HTML(value="<b>Connectivity Profile Parameters:</b>"),
        checkbox_subBox,
        g_cosine_slider,
        w_inh_slider,
    ]

# Initial population
update_connectivity_box()

box_simulation = VBox([
    HTML(value="<b>Simulation Parameters:</b>"),
    input_duration_box,
    velocity_duration_box,
    duration_box,
    angles_dropdown
], layout=Layout(width='25%', align_items='center'))

controls = HBox(
    [box_neurons, box_input, box_connectivity, box_simulation],
    layout=Layout(justify_content='center', margin='20px')
)

# Manual execution button + output area
run_button = Button(
    description='Run Simulation',
    button_style='success',  # green style (builtin)
    icon='play',
    layout=Layout(width='50%', height='55px')
)
run_button.style.font_weight = 'bold'
run_button.style.font_size = '20px'

output_area = Output(layout=Layout(border='1px solid #ccc', padding='10px'))

# Keep upper bound neuron slider in range when number of neurons changes
def on_num_neurons_change(change):
    new_n = change['new']

num_neurons_slider.observe(on_num_neurons_change, names='value')

# Click handler to execute simulation ONLY when button is pressed
def on_run_clicked(b):
    run_button.disabled = True
    try:
        with output_area:
            output_area.clear_output(wait=True)
            print("Running simulation with current parameters...")
            interactive_simulator(
                num_neurons_slider.value,
                tau_slider.value,
                sigma_noise_slider.value,
                stimulus_center_slider.value,
                stimulus_width_slider.value,
                I0_slider.value,
                g_cosine_slider.value,
                velocity_input_slider.value,
                w_inh_slider.value,
                duration_box.value,
                input_duration_box.value,
                velocity_duration_box.value,
                autapse_button.value,
                angles_dropdown.value,
                tau_syn_slider.value,
                Iff_slider.value,
            )
            print("Simulation complete.")
    finally:
        run_button.disabled = False

run_button.on_click(on_run_clicked)

# Assemble dashboard (button on top)
dashboard = VBox([
    HBox([run_button], layout=Layout(justify_content='center')),
    controls,
    output_area
], layout=Layout(align_items='stretch', justify_content='flex-start'))

display(dashboard)